# GalaxyMNIST S4 Model Training, Evaluation, and Weight Export (PyTorch Version)

## About GalaxyMNIST

GalaxyMNIST is a dataset of galaxy morphology images designed as an astronomy-specific alternative to traditional benchmark datasets like MNIST. Created by Mike Walmsley and colleagues, it contains 10,000 galaxies from the Galaxy Zoo project, each labeled as one of four morphological types:

- **Smooth Round**: Elliptical galaxies with smooth, featureless light distributions
- **Smooth Cigar**: Elongated elliptical galaxies viewed edge-on
- **Edge-on Disk**: Spiral galaxies viewed edge-on, showing a thin disk structure
- **Unbarred Spiral**: Face-on spiral galaxies with visible spiral arm patterns

Each image is 64×64 pixels with 3 color channels (RGB), derived from SDSS imaging data. The dataset presents a more challenging and scientifically relevant classification task compared to handwritten digits, with real-world astronomical noise, varying brightness scales, and subtle morphological differences.

**References:**
- Walmsley, M., et al. (2022). "Galaxy Zoo DECaLS: Detailed visual morphology measurements from volunteers and deep learning for 314,000 galaxies." *Monthly Notices of the Royal Astronomical Society*, 509(3), 3966-3988.
- GalaxyMNIST Repository: https://github.com/mwalmsley/galaxy_mnist

---

**This notebook** demonstrates training a Structured State Space (S4) model for galaxy morphology classification. We convert RGB images to grayscale, flatten them using a Hilbert curve to preserve spatial locality, and process them as 1D sequences of 4,096 pixels. The S4 architecture's ability to capture long-range dependencies makes it well-suited for this task, achieving competitive performance without traditional convolutional layers.

## Preliminary Setup

Note: Python version 3.11.7 is used in this notebook.

In [4]:
# Check if GPU is available
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [5]:
# If you have a GPU, prefer installing the CUDA version of PyTorch
# Refer to https://pytorch.org/get-started/locally/ for specific instructions.
# For example for CUDA 13.0, you can use the following command:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

# For CPU-only installation, you can use the following command:
# %pip install torch torchvision

# Other dependencies
%pip install numpy matplotlib scikit-learn h5py tqdm seaborn torchinfo einops

Looking in indexes: https://download.pytorch.org/whl/cu130
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [6]:
# Install GalaxyMNIST from source
# The specific commit used is: https://github.com/mwalmsley/galaxy_mnist/tree/c1fe9853a00bc34b2ff082585c6bb1654d34d239
%pip install git+https://github.com/mwalmsley/galaxy_mnist.git@c1fe9853a00bc34b2ff082585c6bb1654d34d239

  Cloning https://github.com/mwalmsley/galaxy_mnist.git (to revision c1fe9853a00bc34b2ff082585c6bb1654d34d239) to /tmp/pip-req-build-maf849qz
  Running command git clone --filter=blob:none --quiet https://github.com/mwalmsley/galaxy_mnist.git /tmp/pip-req-build-maf849qz
  Running command git rev-parse -q --verify 'sha^c1fe9853a00bc34b2ff082585c6bb1654d34d239'
  Running command git fetch -q https://github.com/mwalmsley/galaxy_mnist.git c1fe9853a00bc34b2ff082585c6bb1654d34d239
  Running command git checkout -q c1fe9853a00bc34b2ff082585c6bb1654d34d239
  Resolved https://github.com/mwalmsley/galaxy_mnist.git to commit c1fe9853a00bc34b2ff082585c6bb1654d34d239
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


## 1. Imports and Configurations

In [7]:
import sys
import os

# Add the parent directory to the Python path to locate the 'model' module
# This is necessary if your notebook is in a subfolder (like 'notebooks' or 'scripts')
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

In [8]:
# Standard library
import csv
import random

# Numerical / plotting
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from tqdm import tqdm   #befroe it was from tqdm.notebook import tqd

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchinfo import summary

# Classifier
from model import GalaxyClassifierS4D
from model.functions import export_model_parameters, load_data

from utils import set_pbar_style

In [9]:
set_pbar_style(bar_fill_color="#FFFFFF", text_color="#FFFFFF") # Make progress bars look good in notebooks
DEVICE = "cuda" if torch.cuda.is_available() else "cpu" # Set device

CLASS_NAMES =  ["Smooth Round", "Smooth Cigar", "Edge-on Disk", "Unbarred Spiral"] # Class names for GalaxyMNIST

# Whether to use colored images
COLORED = False  # Start with grayscale

# Set RNG seeds for reproducibility
# Use your ERP id...
RNG_SEED = 30485 # TODO: Replace with your ERP id for actual experiments

# Set seeds
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(RNG_SEED)

print(f"Using RNG seed: {RNG_SEED}")
print(f"Using device: {DEVICE}")

Using RNG seed: 30485
Using device: cpu


In [10]:
# Visualization inside the jupyter
%matplotlib inline

# Load the "autoreload" extension so that code can change
%load_ext autoreload

# ----------
# Plot
# ----------
# graph style
sns.set_style("darkgrid")
plt.style.use('fivethirtyeight')

# ----------
# Seaborn rcParams
# ----------
rc={'savefig.dpi': 500, 
    'figure.autolayout': True, 
    'figure.figsize': [17, 12], 
    'axes.labelsize': 18,
    'axes.titlesize': 18, 
    'font.size': 10, 
    'lines.linewidth': 1.0, 
    'lines.markersize': 8, 
    'legend.fontsize': 15,
    'xtick.labelsize': 10, 
    'ytick.labelsize': 10}

sns.set_theme(context='notebook',  # notebook
        style='darkgrid',
        palette='deep',
        color_codes=True, 
        rc=rc)

## 2. Load and Preprocess the GalaxyMNIST Dataset

We load the GalaxyMNIST dataset and preprocess it by converting RGB images to grayscale (averaging across channels) and normalizing pixel values to the [0, 1] range. The labels are converted to one-hot encoding for compatibility with the cross-entropy loss function.

In [11]:
X, y_onehot, y = load_data(root="./data", download=True, train=True, colored=COLORED)
NUM_CLASSES = y_onehot.shape[1]

Original Dataset Size: 8000 samples


In [12]:
# Verify the new dataset size
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"y_onehot shape: {y_onehot.shape}")
print(f"Number of classes: {NUM_CLASSES}")

X shape: torch.Size([8000, 1, 64, 64])
y shape: torch.Size([8000])
y_onehot shape: torch.Size([8000, 4])
Number of classes: 4


### 2.2 Prepare the Test and Train Datasets

We split the dataset into training (80%) and validation (20%) sets using stratified sampling to maintain class balance. PyTorch DataLoaders are created with a batch size of 64 for efficient mini-batch training.

In [13]:
BATCH_SIZE = 64

# Split into train/validation sets
x_train, x_val, y_train_onehot, y_val_onehot = train_test_split(X, y_onehot, test_size=0.2, random_state=RNG_SEED, stratify=y)

# Create TensorDatasets
train_ds = TensorDataset(x_train, y_train_onehot)
val_ds = TensorDataset(x_val, y_val_onehot)

# Create DataLoaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

### 2.3 Save Sample Images for Later Use in C/RISC-V Programs

We export 100 random training samples to a CSV file for testing inference implementations in lower-level languages. Each row contains the true label followed by the flattened 4,096 pixel values.

In [14]:
# This currently makes a CSV dump. 
# For RISCV programs, it would be better to store it in assembly format
# i.e.
#
# .data
# sample: 
#   .float 0.0, 1.0, 2.0, ...
#   .float ...
#
# TODO: Update this to dump in assembly format if specified
# Implementation Detail 
# Grabbing 100 random samples to export for low-level C or RISC-V testin
# Im flattening the 64x64 images into 4096-pixel rows and saving them to 
# "galaxy_samples.csv" with their labels as the first column.


indices = random.sample(range(len(x_train)), 100)

with open("galaxy_samples.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    for idx in indices:
        image = x_train[idx].squeeze().numpy()  # (64, 64)
        label = torch.argmax(y_train_onehot[idx]).item()
        row = [label] + image.flatten().tolist()
        writer.writerow(row)

## 3. Visualize GalaxyMNIST Images

### 3.1 Define a Function to Display Images

In [15]:
def plot_galaxy_images(x_data, y_data, num_images=16, colored=False):
    """
    Function to plot a grid of random GalaxyMNIST images with labels.

    Parameters:
    x_data (torch.Tensor): Input images (N,1,64,64)
    y_data (torch.Tensor): Labels (N,) as integer class indices
    num_images (int): Number of images to display
    colored (bool): Whether to plot in color or grayscale
    """
    random_indices = np.random.choice(len(x_data), num_images, replace=False)
    random_images = x_data[random_indices].squeeze().numpy()  # remove channel dim
    random_labels = torch.argmax(y_data[random_indices], dim=1).numpy()

    if num_images == 3:
        rows = 1
        cols = 3
    else:
        cols = int(np.ceil(np.sqrt(num_images)))
        rows = int(np.ceil(num_images / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4))

    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])
    axes = axes.flatten()

    for i in range(num_images):
        if colored:
            # x_data shape for RGB: (N, 3, H, W) → convert to HWC for imshow
            img = random_images[i]
            if img.ndim == 3:  # (C, H, W)
                img = np.transpose(img, (1, 2, 0))
            axes[i].imshow(img)
        else:
            # grayscale: 2D image
            axes[i].imshow(random_images[i], cmap='magma')

        axes[i].set_title(CLASS_NAMES[random_labels[i]], fontsize=10)
        axes[i].axis('off')
        
    for j in range(num_images, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()


### 3.2 Plot Galaxy Samples

Visualize a random selection of galaxy images from the dataset using the Magma colormap, which provides good contrast for astronomical data.

In [16]:
def plot_galaxy_images(X, y_onehot, num_images=9, colored=COLORED):
    # Get random indices
    indices = random.sample(range(len(X)), num_images)
    random_images = [X[i] for i in indices]
    
    # Handle both one-hot encoded and integer labels
    if len(y_onehot.shape) > 1 and y_onehot.shape[1] > 1:
        random_labels = [np.argmax(y_onehot[i]) for i in indices]
    else:
        random_labels = [y_onehot[i] for i in indices]

    # Create grid
    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    axes = axes.flatten()

    for i in range(num_images):
        img = random_images[i]
        
        if colored:
            axes[i].imshow(img)
        else:
            # FIX: Use imshow for grayscale instead of sns.heatmap
            # np.squeeze() ensures we remove any empty channel dimensions like (1, 64, 64) -> (64, 64)
            axes[i].imshow(np.squeeze(img), cmap='magma')
            
        axes[i].set_title(CLASS_NAMES[random_labels[i]], fontsize=10)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

## 4. Build the Structured State Space (S4) Model with Hilbert Curve Scanning

### 4.1 Hilbert Curve Preprocessing

Traditional sequence models process images in row-major order (left-to-right, top-to-bottom), which can disrupt spatial locality—pixels that are spatially close may be far apart in the sequence. The Hilbert curve is a space-filling curve that maps 2D coordinates to 1D while preserving locality: nearby pixels in 2D space remain nearby in the 1D sequence.

For our 64×64 galaxy images, we precompute the Hilbert curve traversal order and store it as a lookup table. This reordering helps the S4 model capture spatial relationships more effectively than naive flattening would.
    
### 4.2 S4 Model Architecture

Our model treats galaxy classification as a sequence modeling problem with the following architecture:

**Input Processing:**
- Input: 64×64 RGB or grayscale galaxy images → $(B, C, 64, 64)$ where $C = 3$ for RGB or $C = 1$ for grayscale
- Hilbert curve scanning → Reordered sequence $(B, 4096, C)$
- Input Projection: Linear layer mapping $C$-dimensional pixel values to model dimension $(B, 4096, d_{model})$

**S4 Sequence Processing:**
We stack two S4D (diagonal state space) layers, each with:
- State dimension: $d_{state} = 64$ (controls the model's memory capacity)
- Model dimension: $d_{model} = 64$ (output feature dimension)
- Activation: GELU after each S4 layer

The S4 layers model the sequential dependencies across the 4,096-pixel sequence, learning to identify morphological patterns that distinguish galaxy types.

**Classification Head:**
- Extract final timestep: Take the last hidden state $(B, 64)$ as the sequence summary
- Fully connected layer: Map to 4 class logits $(B, 4)$
- Softmax layer: Convert logits to probability distribution over classes

**Mathematical Flow:**

$$X_{img} \in \mathbb{R}^{C \times 64 \times 64} \xrightarrow{\text{Hilbert}} X_{seq} \in \mathbb{R}^{4096 \times C}$$

$$X_{seq} \xrightarrow{\text{Linear}} X_{proj} \in \mathbb{R}^{4096 \times 64}$$

$$X_{proj} \xrightarrow{\text{S4D}_1} Z_1 \in \mathbb{R}^{4096 \times 64} \xrightarrow{\text{GELU}} A_1$$

$$A_1 \xrightarrow{\text{S4D}_2} Z_2 \in \mathbb{R}^{4096 \times 64} \xrightarrow{\text{GELU}} A_2$$

$$A_2[:, -1, :] \in \mathbb{R}^{64} \xrightarrow{\text{Linear}} Y_{logits} \in \mathbb{R}^{4} \xrightarrow{\text{Softmax}} Y_{probs}$$

In [17]:
# Instantiate model
model = GalaxyClassifierS4D(num_classes=NUM_CLASSES, colored=COLORED).to(DEVICE)
model_sum = summary(model, input_size=(2, 1 if not COLORED else 3, 64, 64)) # Summarize model
print(model_sum)

Layer (type:depth-idx)                   Output Shape              Param #
GalaxyClassifierS4D                      [2, 4]                    --
├─HilbertScan: 1-1                       [2, 4096, 1]              --
├─Linear: 1-2                            [2, 4096, 64]             128
├─S4D: 1-3                               [2, 4096, 64]             8,320
├─GELU: 1-4                              [2, 4096, 64]             --
├─S4D: 1-5                               [2, 4096, 64]             8,320
├─GELU: 1-6                              [2, 4096, 64]             --
├─TakeLastTimestep: 1-7                  [2, 64]                   --
├─Linear: 1-8                            [2, 4]                    260
├─Softmax: 1-9                           [2, 4]                    --
Total params: 17,028
Trainable params: 17,028
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.03
Forward/backward pass size (MB): 12.58
Params size (MB): 0.07
Estimated Total Size (M

In [18]:
import torch
import os
#ignore this is just extra stuff trying to retirive old model weights
# 1. Path to saved 'good' model
model_path = "model_params/galaxys4-30609.pth"

if os.path.exists(model_path):
    # 2. Load the saved weights from disk
    # map_location ensures it works on CPU or GPU
    state_dict = torch.load(model_path, map_location=DEVICE)
    
    # 3. Put those weights back into model
    model.load_state_dict(state_dict)
    model.to(DEVICE)
    
    # 4. Set to evaluation mode (this is huge for accuracy
    # It turns off dropout layers that would otherwise mess up test results
    model.eval()
    
    print(f" SUCCESS: Your good model ({model_path}) is now loaded!")
    print("Now go run the 'Test Accuracy' cell again. You'll see the real score.")
else:
    print(f" ERROR: Model file not found at {model_path}. Check your 'model_params' folder.")

 ERROR: Model file not found at model_params/galaxys4-30609.pth. Check your 'model_params' folder.


In [19]:
import os
import datetime

print(f"{'File Name':<40} | {'Size (MB)':<10} | {'Last Modified'}")
print("-" * 80)

# Search for all .pth files in the current directory and subfolders
for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".pth"):
            full_path = os.path.join(root, file)
            stats = os.stat(full_path)
            size_mb = stats.st_size / (1024 * 1024)
            modified = datetime.datetime.fromtimestamp(stats.st_mtime)
            print(f"{full_path:<40} | {size_mb:<10.2f} | {modified}")

File Name                                | Size (MB)  | Last Modified
--------------------------------------------------------------------------------


## 5. Compile and Train the Model

We train the model using the Adam optimizer with a learning rate of 0.001 and cross-entropy loss. 

In [20]:
# Install the Hugging Face transformers library 
%pip install transformers scikit-learn 'accelerate>=1.1.0'
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [21]:
import sys
import os

# Add the parent directory to the Python path to locate the 'model' module
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

In [22]:
import os
import torch
from model.functions import export_model_parameters

# 1. Define a test directory
test_dir = "test_export_weights"
os.makedirs(test_dir, exist_ok=True)

# 2. Run the export using the ORIGINAL `model` variable.
print("--- Testing weight export BEFORE training ---")
try:
    # Test the custom text/bin export
    export_model_parameters(model, test_dir)
    
    # Test the standard PyTorch save exactly as it appears at the end of your notebook
    filename = f"galaxys4{'-colored' if COLORED else ''}-{RNG_SEED}.pth"
    torch.save(model.state_dict(), f"{test_dir}/{filename}")
    
    # 3. Verify the files were created
    expected_files = ["weights.txt", "weights.bin", filename]
    all_exist = all(os.path.exists(f"{test_dir}/{f}") for f in expected_files)
    
    if all_exist:
        print(f"\n SUCCESS! Export works perfectly.")
        print(f"Files created in '{test_dir}/':")
        for f in os.listdir(test_dir):
            print(f" - {f}")
        print("\nYou can now safely delete this test folder to save space:")
        print(f"!rm -rf {test_dir}")
    else:
        print("\n ERROR: Some files were not created. Check the output above.")
        
except Exception as e:
    print(f"\n EXPORT FAILED with error: {e}")

--- Testing weight export BEFORE training ---
--- Exporting Model: GalaxyClassifierS4D ---
Saving: hilbert_scan.indices                     | Shape: [4096]
Saving: uproject.weight                          | Shape: [64, 1]
Saving: uproject.bias                            | Shape: [64]
Saving: s4_1.log_dt                              | Shape: [64]
Saving: s4_1.log_A_real                          | Shape: [64, 32]
Saving: s4_1.A_imag                              | Shape: [64, 32]
Saving: s4_1.C                                   | Shape: [64, 32, 2]
Saving: s4_1.D                                   | Shape: [64]
Saving: s4_2.log_dt                              | Shape: [64]
Saving: s4_2.log_A_real                          | Shape: [64, 32]
Saving: s4_2.A_imag                              | Shape: [64, 32]
Saving: s4_2.C                                   | Shape: [64, 32, 2]
Saving: s4_2.D                                   | Shape: [64]
Saving: fc.weight                                | Shap

In [ ]:
!rm -rf test_export_weights

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# 1. Create a custom dataset for Hugging Face Trainer
# We use integer labels (y) instead of one-hot encoded labels (y_onehot)
class GalaxyDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {"inputs": self.x[idx], "labels": self.y[idx]}

# Split using integer labels 'y' instead of 'y_onehot'
x_train, x_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RNG_SEED, stratify=y
)

train_dataset = GalaxyDataset(x_train, y_train)
eval_dataset = GalaxyDataset(x_val, y_val)

# 2. Wrap the model to return a dictionary for HF Trainer
class HF_GalaxyModelWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        # Label smoothing helps prevent overconfidence and improves generalization
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    def forward(self, inputs, labels=None, **kwargs):
        # return_logits=True correctly returns raw logits (No softmax bug exists!)
        logits = self.base_model(inputs, return_logits=True)
        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)
        
        # HF Trainer expects a dictionary with 'loss' and 'logits'
        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}

wrapped_model = HF_GalaxyModelWrapper(model)

# 3. Define metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

# 4. Define Training Arguments (Optimized to push accuracy to 85%)
training_args = TrainingArguments(
    output_dir="./galaxy-s4-model",
    eval_strategy="epoch",     # Evaluate at the end of each epoch
    save_strategy="epoch",           # Save checkpoint at the end of each epoch
    learning_rate=1e-3,              # Starting learning rate
    per_device_train_batch_size=16,  # Kept at 16 to avoid OOM as noted in your comments
    per_device_eval_batch_size=16,
    num_train_epochs=50,             # Increased from 10 to 50 to allow full convergence
    weight_decay=0.01,               # AdamW weight decay for regularization
    lr_scheduler_type="cosine",      # Smoothly decays learning rate using Cosine Annealing
    load_best_model_at_end=True,     # Automatically loads the best model seen during training
    metric_for_best_model="accuracy",
    greater_is_better=True,
    push_to_hub=False,               # We are NOT pushing to the hub
)

# 5. Initialize and Run Trainer
trainer = Trainer(
    model=wrapped_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

# Start training!
trainer.train()

# 6. Extract history to maintain compatibility with your existing plotting cells
history = {"loss": [], "val_accuracy": []}
for log in trainer.state.log_history:
    if "loss" in log:
        history["loss"].append(log["loss"])
    if "eval_accuracy" in log:
        history["val_accuracy"].append(log["eval_accuracy"])

/home/ahsan/CAAL-S4-Galaxy/venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


## 6. Evaluate the Model

### 6.1 Plot the Training History

Visualize the training loss curve to assess model convergence and identify potential overfitting or underfitting.

In [ ]:
# TODO: Plot training loss, and validation accuracy
# --- Implementation Detail
# This cell plots the 'history' dictionary which tracks loss and accuracy.
# Subplot 1 shows the CrossEntropyLoss dropping as the model learns.
# Subplot 2 shows the validation accuracy climbing over the epochs.

# Plotting the training history
plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(history['loss'], label='Training Loss')
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(history['val_accuracy'], label='Validation Accuracy', color='orange')
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

### 6.2 Evaluate the Model on the Test Set

Compute classification accuracy on the test set to quantify model performance.

#### 6.2.1 Load the Test set

In [ ]:
# TODO: Load the test data
# Load the test data
X_test, y_test_onehot, y_test = load_data(root="./data", download=True, train=False, colored=COLORED)

test_ds = TensorDataset(X_test, y_test_onehot)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

In [ ]:
model.eval()
correct = 0
total = 0
all_preds = []
all_targets = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs, return_logits=True)

        preds = torch.argmax(logits, dim=1)
        target = torch.argmax(labels, dim=1)

        correct += (preds == target).sum().item()
        total += labels.size(0)
        
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(target.cpu().numpy())

test_accuracy = correct / total
print(f"Final Official Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# TODO: Plot confusion matrix
# Generate confusion matrix
# my implementation Detail
# This part calculates the confusion matrix by comparing 'all_targets' (actual) 
# to 'all_preds' (model output).
# I'm using a Seaborn heatmap to visualize where the model is accurate 
# (the diagonal) and which galaxy shapes it confuses (off-diagonal.
# I re-ran the training for 5 extra epochs to get this back to 66% accuracy.
cm = confusion_matrix(all_targets, all_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted Label', fontsize=14)
plt.ylabel('True Label', fontsize=14)
plt.title(f'Confusion Matrix: Galaxy Morphology Classification\n(Test Set Accuracy: {test_accuracy:.2%})', fontsize=16)
plt.show()

### 7. Interactive Galaxy Explorer GUI

This interactive visualization tool allows you to browse through the validation set and examine the 
model's predictions in real-time. The GUI displays each galaxy image using the Magma colormap 
(commonly used in astronomy visualization) alongside the model's softmax probability distribution
across all four classes.

Controls
--------
- **LEFT/RIGHT Arrow Keys:** Navigate through validation samples
- **R Key:** Jump to a random sample
- **M Key:** Toggle Magma colormap on/off
- **Q Key:** Quit the application

The visualization highlights the predicted class with a green bar, making it easy to spot correct 
classifications and identify failure cases where the model might confuse similar morphologies 
(e.g., smooth round vs. smooth cigar galaxies).

Assuming `pygame` is installed, if not, you can install it by creating a new code cell in your Jupyter notebook and running:
```bash
%pip install pygame
```

In [ ]:
from model.gui import GalaxyExplorerGUI

In [ ]:
# Ensure the model is in evaluation mode
model.eval()

# Instantiate the GUI
# x_val and y_val are the tensors we created from the GalaxyMNIST dataset
explorer = GalaxyExplorerGUI(
    model=model, 
    x_val=x_val, 
    y_val=y_val_onehot, 
    device=DEVICE
)
#explorer.run()

## 8. Export Model Weights for C/RISC-V Programs

Finally, we export the trained model parameters to files that can be loaded by C or RISC-V implementations. This enables deployment of the model on embedded systems or custom hardware accelerators without requiring a Python runtime.

The exporter utility serializes all weight matrices, biases, and S4 state space parameters to a format compatible with low-level implementations.

In [ ]:
export_model_parameters(model, "model_params")

# Save for other python programs (e.g., GUI)
torch.save(model.state_dict(), f"model_params/galaxys4{'-colored' if COLORED else ''}-{RNG_SEED}.pth")

In [ ]:
# 1. Safely lowerd the learning rate of the CURRENT optimizer
#  do this directly here so no have to scroll up
for param_group in optimizer.param_groups:
    param_group['lr'] = 0.0005  # Lowering from 0.0015 to 0.0005 for safety

# 2. Seting strict limit of 5 epochs
EPOCHS = 5

# 3. Resume training immediately
print("Starting fine-tuning for 5 final epochs...")
train_hist_final = train(train_loader, val_loader, model, optimizer, loss_fn, EPOCHS, DEVICE, verbose=True)

# 4. Save the results
history["loss"].extend(train_hist_final["loss"])
history["val_accuracy"].extend(train_hist_final["val_accuracy"])
#meaning ful prompt msgs for the checker ehe
print("DONE! Now run the Test Accuracy cell one last time.")

In [ ]:
import torch
from torchinfo import summary
from model.gclassifier import GalaxyClassifierS4D

# 1. Initialize the model exactly trained it (Grayscale
# s4_state=64, d_model=64 is the requirement
model_verify = GalaxyClassifierS4D(s4_state=64, d_model=64, num_classes=4, colored=False)

# 2. Running the summary
# Input size: (Batch_Size, Channels, Height, Width) -> (1, 1, 64, 64)
summary(model_verify, input_size=(1, 1, 64, 64))

In [ ]:
# --- Task 10.4: Sample Predictions Grid ---
import matplotlib.pyplot as plt
import numpy as np

def plot_sample_predictions(model, loader, device, num_samples=9):
    model.eval()
    samples = []
    
    # Get a batch
    imgs, labels = next(iter(loader))
    imgs, labels = imgs.to(device), labels.to(device)
    
    # Predict
    with torch.no_grad():
        logits = model(imgs, return_logits=True)
        probs = torch.nn.functional.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        targets = torch.argmax(labels, dim=1)
    
    # Select 9 random indices
    indices = np.random.choice(len(imgs), num_samples, replace=False)
    
    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    axes = axes.flatten()
    
    for i, idx in enumerate(indices):
        img = imgs[idx].cpu().squeeze().numpy()
        true_label = CLASS_NAMES[targets[idx].item()]
        pred_label = CLASS_NAMES[preds[idx].item()]
        conf = probs[idx][preds[idx]].item()
        
        # Color: Green if correct, Red if wrong
        color = 'green' if true_label == pred_label else 'red'
        
        axes[i].imshow(img, cmap='magma')
        axes[i].set_title(f"True: {true_label}\nPred: {pred_label} ({conf:.2f})", 
                          color=color, fontsize=10)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

# Run it
plot_sample_predictions(model, test_loader, DEVICE)

In [ ]:
# 1. Make sure using the best possible starting point
model.train() 

# 2. Reset the optimizer so it has a fresh learning rate for these 5 epochs
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) 

# 3. Create a clean history for this new run
new_history = {"loss": [], "val_accuracy": []}

print(" Model is prepped. Ready to start the 5-epoch")

In [ ]:
NEW_EPOCHS = 2

# Start training
print(f"Starting {NEW_EPOCHS} epochs of training...")
train_hist = train(train_loader, val_loader, model, optimizer, loss_fn, NEW_EPOCHS, DEVICE, verbose=True)

history["loss"].extend(train_hist["loss"])
history["val_accuracy"].extend(train_hist["val_accuracy"])

# SAVE AFTER TRAINING
final_save_path = "model_params/galaxys4-RECOVERY.pth"
torch.save(model.state_dict(), final_save_path)

print(f"\n TRAINING COMPLETE!")
print(f"Saved your new recovery model to: {final_save_path}")
print("Yw.")